In [0]:
from pyspark.sql.functions import *

In [0]:
storage_account= "storageretaillakehouse"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
   "<YOUR KEY>"
)


In [0]:
fact_sales_path = f"abfss://gold@storageretaillakehouse.dfs.core.windows.net/facts/fact_sales"
dim_dates_path = f"abfss://gold@storageretaillakehouse.dfs.core.windows.net/dimensions/dim_dates"

In [0]:
fact_sales_df= spark.read.format('delta').load(fact_sales_path)
dim_dates_df= spark.read.format('delta').load(dim_dates_path)


In [0]:
fact_sales_df.printSchema()
dim_dates_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- customer_key: long (nullable = true)
 |-- product_key: long (nullable = true)
 |-- date_key: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- total_payment_value: double (nullable = true)
 |-- total_installments: long (nullable = true)
 |-- sale_key: long (nullable = true)

root
 |-- date: date (nullable = true)
 |-- date_key: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- month_name: string (nullable = true)
 |-- week: integer (nullable = true)
 |-- day: integer (nullable = true)



In [0]:
monthly_base = fact_sales_df.alias("f").join(dim_dates_df.alias("d"), on="date_key", how="left")

In [0]:
monthly_base.printSchema()

root
 |-- date_key: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- customer_key: long (nullable = true)
 |-- product_key: long (nullable = true)
 |-- order_status: string (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- total_payment_value: double (nullable = true)
 |-- total_installments: long (nullable = true)
 |-- sale_key: long (nullable = true)
 |-- date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- month_name: string (nullable = true)
 |-- week: integer (nullable = true)
 |-- day: integer (nullable = true)



In [0]:
monthly_sales = monthly_base.groupBy("year", "month", "month_name").agg(countDistinct("order_id").alias("total_orders"), sum("total_payment_value").alias("revenue"))

In [0]:
monthly_sales = monthly_sales.orderBy("year", "month")

In [0]:
display(monthly_sales)

year,month,month_name,total_orders,revenue
2016,9,September,3,347.52
2016,10,October,308,73914.58
2016,12,December,1,19.62
2017,1,January,789,187779.40999999997
2017,2,February,1733,344134.7899999998
2017,3,March,2641,526961.6599999996
2017,4,April,2391,505665.5299999997
2017,5,May,3660,724504.55
2017,6,June,3217,600753.2699999999
2017,7,July,3969,737293.0800000001


In [0]:
from common_scripts.dq_framework import *

In [0]:
run_dq_check(
    monthly_sales,
    "monthly_sales",
    "null_total_revenue",
    col("revenue").isNull(),
    severity="critical"
)

run_dq_check(
    monthly_sales,
    "monthly_sales",
    "null_total_orders",
    col("total_orders").isNull(),
    severity="critical"
)


[CRITICAL] monthly_sales | null_total_revenue: 0
[CRITICAL] monthly_sales | null_total_orders: 0


0

In [0]:
dq_df = dq_df_from_dq_results(spark, dq_results)

In [0]:
dq_path = f"abfss://audit@{storage_account}.dfs.core.windows.net/gold_dq_logs/kpi_monthly_sales"

In [0]:
dq_df.write.format('delta').mode('append').save(dq_path)

In [0]:
monthly_sales_path = f"abfss://gold@{storage_account}.dfs.core.windows.net/kpi/kpi_monthly_sales"


monthly_sales.write.format('delta').mode('overwrite').save(monthly_sales_path)

In [0]:
month_sales = spark.read.format('delta').load(monthly_sales_path)
display(month_sales)


year,month,month_name,total_orders,revenue
2016,9,September,3,347.52
2016,10,October,308,73914.58
2016,12,December,1,19.62
2017,1,January,789,187779.40999999997
2017,2,February,1733,344134.7899999998
2017,3,March,2641,526961.6599999996
2017,4,April,2391,505665.5299999997
2017,5,May,3660,724504.55
2017,6,June,3217,600753.2699999999
2017,7,July,3969,737293.0800000001
